# AI 2 · 2주차 전처리 비교 실험 (명암·회전·노이즈 보정)

2주차에 추가한 **명암·회전·노이즈 보정**이 OCR을 얼마나 개선하는지, 1주차 방식과 나란히 비교합니다.

| 방식 | 설명 |
|---|---|
| `none` | 전처리 없음 |
| `v1` | 1주차 조합 (조건별) |
| `v2` | 2주차 조합 (조건별) |
| `auto` | 실제 서비스 흐름(`pipeline.run_pipeline`): 품질 진단으로 전처리를 고르고, OCR 확신도가 낮을 때만 회전·뒤집힘을 다시 확인 |

**실행 방법:** 메뉴에서 **런타임 → 모두 실행**을 누르면 됩니다. 사진 24장 기준 **30분에서 1시간** 걸립니다.

- 실행하는 동안 Colab 탭은 닫지 말고, 노트북이 절전 모드로 들어가지 않게 해 주세요.
- 끝나면 마지막 셀에서 결과 압축 파일(`ai2_week2_paddle.zip`)이 내려받아집니다.
- 압축을 푼 내용물은 GitHub의 `results/ai2_week2/paddle/` 폴더에 올리면 됩니다.

## 1. 레포 받아오기

GitHub에서 코드나 사진을 고쳤다면 이 셀부터 다시 실행해야 바뀐 내용이 반영됩니다.

In [ ]:
REPO_URL = "https://github.com/DL-2026F-Team05/package-ocr-preprocessing.git"
OCR_ENGINE = "paddle"   # 설치나 실행 오류가 나면 "easyocr"로 바꾸고 이 셀부터 다시 실행

import os, subprocess, sys

os.chdir("/content")
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")   # 비공개 레포일 때만 필요
except Exception:
    token = None
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if not os.path.exists("package-ocr"):
    if subprocess.run(["git", "clone", "-q", url, "package-ocr"]).returncode != 0:
        raise RuntimeError("레포를 받아오지 못했습니다. REPO_URL과 공개/비공개 설정(토큰)을 확인하세요.")
os.chdir("/content/package-ocr")
subprocess.run(["git", "pull", "-q"])
print("현재 위치:", os.getcwd())

## 2. 패키지 설치

`koreanize-matplotlib`는 결과 그림의 한글이 깨지지 않게 해 줍니다.

In [ ]:
pkgs = ["paddlepaddle", "paddleocr"] if OCR_ENGINE == "paddle" else ["easyocr"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs, "koreanize-matplotlib"], check=True)
print("설치 완료:", ", ".join(pkgs + ["koreanize-matplotlib"]))

## 3. 사진과 정답 라벨 확인

In [ ]:
from collections import Counter
from pathlib import Path

sys.path.insert(0, "preprocessing")
import experiment, preprocess, pipeline

RAW, GEN, OUT = Path("data/raw"), Path("data/generated"), Path("results/ai2_week2/paddle")

raw_images, skipped = experiment.list_images(RAW)
labels = experiment.load_labels("data/labels/labels.csv")
print(f"직접 찍은 사진 {len(raw_images)}장, 정답 라벨이 있는 제품 {len(labels)}개")
if skipped:
    print("[건너뜀] 이름 규칙에 맞지 않는 파일:", ", ".join(skipped))

## 4. 조건 사진 만들기

정상(C0) 사진에서 저조도(C1), 흔들림(C3), **회전(C6, 15°/45°/90°/180°)**, **노이즈(C10)** 사진을 만듭니다. 회전과 노이즈는 2주차에 새로 추가한 조건입니다.

In [ ]:
generated = experiment.generate_from_clean(RAW, GEN, conditions=("c1", "c3", "c6", "c10"))
images = raw_images + generated
counts = Counter(experiment.parse_name(p)[1] for p in images)
print(f"전체 {len(images)}장")
for cond in sorted(counts, key=lambda c: int(c[1:])):
    print(f"  {cond} {experiment.COND_NAMES.get(cond, '')}: {counts[cond]}장")

## 5. 사진 품질 진단 미리보기

`assess_quality`가 사진마다 문제를 찾아내고, 다시 찍어야 하는지와 추천 전처리를 정합니다. OCR 없이 계산만 해서 금방 끝납니다.

In [ ]:
import pandas as pd

rows = []
for p in images:
    q = preprocess.assess_quality(preprocess.load_image(p))
    rows.append({"file": p.name, "issues": ",".join(q["issues"]) or "-", "retake": q["retake"],
                 "steps": "+".join(q["steps"]) or "-", "advice": q["advice"]})
pd.set_option("display.max_colwidth", 60)
pd.DataFrame(rows)

## 6. OCR 불러오기

In [ ]:
from ocr_eval import OCREngine

ocr = OCREngine(OCR_ENGINE)
print("OCR 준비 완료:", OCR_ENGINE)

## 7. 비교 실험 (30분에서 1시간)

사진마다 `none`, `v1`, `v2`, `auto` 네 가지 방식으로 OCR을 돌립니다. 방식끼리 전처리 조합이 같으면 OCR은 한 번만 돌립니다. 회전 보정은 후보 각도마다 작은 사진으로 OCR을 돌려서 가장 잘 읽히는 방향을 고릅니다. `auto`는 OCR 확신도가 낮은 사진에서만 방향을 확인합니다.

`[1/24]`부터 `[24/24]`까지 한 줄씩 늘어나면 정상입니다.

In [ ]:
df = experiment.run_comparison(images, ocr, labels, OUT)
print("\n저장 위치:", OUT)

## 8. 결과 표와 그림

- 숫자는 평균 CER입니다. **낮을수록 좋습니다.**
- 같은 줄에서 `none`보다 숫자가 작으면 그 전처리가 효과가 있었다는 뜻입니다.

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown("**조건별 평균 CER (유통기한·제품명·알레르기 평균)**"))
display(experiment.comparison_table(df, "cer_mean"))
display(Markdown("**조건별 알레르기 성분 CER**"))
display(experiment.comparison_table(df, "cer_allergen"))
display(Image(str(OUT / "cer_by_variant.png"), width=700))

## 9. 비교 이미지 보기

조건별로 한 장씩 보여 줍니다. 전체 이미지는 `results/ai2_week2/paddle/compare/`에 있습니다.

In [ ]:
import cv2
import matplotlib.pyplot as plt

shown = set()
for path in sorted((OUT / "compare").glob("*.jpg")):
    cond = experiment.parse_name(path)[1]
    if cond in shown:
        continue
    shown.add(cond)
    plt.figure(figsize=(16, 5))
    plt.imshow(cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB))
    plt.title(f"{path.name} ({cond})")
    plt.axis("off")
    plt.show()

## 10. 모듈 연결 샘플 만들기

다른 팀원이 바로 쓸 수 있도록, 사진 → 품질 진단 → 전처리 → OCR 결과를 `docs/interface.md` 형식의 JSON으로 저장합니다.

- **AI 3 (정보 추출):** JSON의 `ocr` 목록을 `extract()`의 입력으로 쓰면 됩니다.
- **AI 4 (TTS·웹 데모):** `quality.retake`가 `true`면 `quality.advice` 문장을 음성으로 읽어 주면 됩니다.

In [ ]:
samples = [RAW / "p01_c0_1.jpg", RAW / "p01_c1_1.jpg", GEN / "p01_c6_L2.jpg", GEN / "p01_c6_L4.jpg"]
for p in samples:
    if not p.exists():
        continue
    data = {"image": p.name, "engine": OCR_ENGINE, **pipeline.run_pipeline(preprocess.load_image(p), ocr)}
    pipeline.save_json(data, OUT / "sample_output" / f"{p.stem}.json")
    q = data["quality"]
    print(f"{p.name}: 글자 {len(data['ocr'])}개, 전처리 {data['preprocess']['steps'] or '없음'}, "
          f"재촬영 {'필요 - ' + q['advice'] if q['retake'] else '불필요'}")

## 11. 결과 내려받기

`ai2_week2_paddle.zip`이 내려받아집니다. 다운로드가 안 됐으면 이 셀만 다시 실행하세요.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/ai2_week2_paddle", "zip", OUT)
files.download("/content/ai2_week2_paddle.zip")